In [ ]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

In [ ]:
### +++ LOAD CLIMATE DATA +++ ###

diags=['h200','h700']
dat={ 'merra2':{}, 'ctrl':{}, 'max.global':{} }

# number of years to omit 
omit_yrs = 30
n_omit = omit_yrs * 12
tmax = 350 * 12

# calculate annual mean fields
for key in dat.keys():
    for varn in diags:
        if key=='merra2':
            dat[key][varn] = xr.open_dataset(f'obs.merra2.{varn}.nc')[varn].mean(dim='time')
        else:
            dat[key][varn] = xr.open_dataset(f'aogcm.{key}.{varn}.nc')[varn].isel(time=slice(n_omit,tmax)).mean(dim='time')

# calculate stationary wave (height anomaly) by subtracting time-mean-zonal-mean from height field
z700={}
z200={}
for key in dat.keys()
    z700[key]=dat[key]['z700']-dat[key]['z700'].mean(dim='lon')
    z200[key]=dat[key]['z200']-dat[key]['z200'].mean(dim='lon')

In [ ]:
text_kw={'weight':'bold', 'ha':'center', 'va':'center', 'color':'black', 'size':14}
# colormap
cmap=cm.RdBu_r
vmin=-100
vmax=100
levels=np.linspace(vmin, vmax, 21)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
# map specs
lon=z700['ctrl'].lon
lat=z700['ctrl'].lat
proj=ccrs.PlateCarree()
trans=ccrs.PlateCarree()

fig,ax=plt.subplots(nrows=2, ncols=2, figsize=(10, 5.25), layout='constrained', subplot_kw={'projection': proj})

ax[0,0].text(-225,0,'OBS $-$ CTRL', rotation=90, **text_kw)
ax[1,0].text(-225,0,'MAX$_{global}$ $-$ CTRL', rotation=90, **text_kw)
ax[0,0].text(0,100,'700hPa', **text_kw)
ax[0,1].text(0,100,'200hPa', **text_kw)

# OBS-CTRL
ax[0,0].pcolormesh(lon, lat, z700['merra2']-z700['ctrl'], cmap=cmap, norm=norm, transform=trans)
ax[0,1].pcolormesh(lon, lat, z200['merra2']-z200['ctrl'], cmap=cmap, norm=norm, transform=trans)

# MAX-CTRL
ax[1,0].pcolormesh(lon, lat, z700['max']-z700['ctrl'], cmap=cmap, norm=norm, transform=trans)
cf=ax[1,1].pcolormesh(lon, lat, z200['max']-z200['ctrl'], cmap=cmap, norm=norm, transform=trans)

letters=['a.','b.','c.','d.']
for i, ax in enumerate(ax.flat):
    ax.text(167,-82,letters[i], **text_kw)
    # map properties
    ax.coastlines(color='k')
    ax.set_global()
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    if i==0:
        gl.bottom_labels=False; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    if i==1:
        gl.bottom_labels=False; gl.left_labels=False; gl.top_labels=False; gl.right_labels=False
    if i==2:
        gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    if i==3:
        gl.bottom_labels=True; gl.left_labels=False; gl.top_labels=False; gl.right_labels=False
        
# add colorbar
cax=fig.add_axes([1.005, 0.2, 0.02, 0.6])
cbar=fig.colorbar(cf, orientation='vertical', extend='both', cax=cax)
cbar.set_label('[m]', labelpad=10, rotation=270, size=10, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=10)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')